# Phase 1 — Elo Model Training 

Build an Elo rating system from 49,450 international football matches (1872–2026).
The end result is a rating for every national team that has ever played, calibrated
on tournament importance and venue.

**Outputs saved to `data/processed/`:**
- `matches_with_elo.csv` — every match with pre-match Elo for both sides (used in backtesting)
- `elo_ratings.json` — current Elo for all 326 teams (used in Monte Carlo)

---

Aufbau eines Elo-Bewertungssystems aus 49.450 internationalen Fußballspielen
(1872–2026). Das Ergebnis ist eine Bewertung für jede Nationalmannschaft, die je
gespielt hat, kalibriert nach Turnierwichtigkeit und Spielort.

**Ausgaben gespeichert in `data/processed/`:**
- `matches_with_elo.csv` — jedes Spiel mit Vor-Spiel-Elo für beide Seiten (verwendet beim Backtesting)
- `elo_ratings.json` — aktuelle Elo-Werte für alle 326 Mannschaften (verwendet in Monte Carlo)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
results = pd.read_csv("../data/raw/results.csv", parse_dates=["date"])

print(f"Total matches: {len(results):,}")
print(f"Date range: {results['date'].min().date()} - {results['date'].max().date()}")
print(f"Columns: {list(results.columns)}")
results.head()

Total matches: 49,450
Date range: 1872-11-30 - 2026-06-27
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## 1. What's in the data / Ein erster Blick auf die Daten

A first look at the source: which tournament types dominate, how matches are
distributed across decades, and how many unique national teams have ever played.

---

Ein erster Blick auf die Quelle: welche Turniertypen dominieren, wie die Spiele
über die Jahrzehnte verteilt sind und wie viele verschiedene Nationalmannschaften
je gespielt haben.

In [3]:
# Tournament type distribution (top 10)
print("Top 10 tournament types:")
print(results['tournament'].value_counts().head(10))

print("\nMatches per decade (last 6 decades):")
results['decade'] = (results['date'].dt.year // 10) * 10
print(results['decade'].value_counts().sort_index().tail(6))

print(f"\nUnique teams ever played: {pd.concat([results['home_team'], results['away_team']]).nunique()}")
print(f"Matches at neutral venues: {results['neutral'].sum():,} ({results['neutral'].mean()*100:.1f}%)")

Top 10 tournament types:
tournament
Friendly                                18368
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64

Matches per decade (last 6 decades):
decade
1970    4133
1980    5025
1990    6944
2000    9526
2010    9787
2020    6075
Name: count, dtype: int64

Unique teams ever played: 336
Matches at neutral venues: 13,100 (26.5%)


## 2. Filter to modern era and assign tournament weights / Filterung auf moderne Ära und Vergabe von Turniergewichten

We restrict to matches from 1990 onwards, older data is sparse and not predictive
of today's strength. Each match also gets a **K-factor** based on tournament
importance: a World Cup match moves Elo ratings 3× more than a friendly.
Follows the [World Football Elo Ratings](https://www.eloratings.net/about) convention.

---

Wir beschränken uns auf Spiele ab 1990, ältere Daten sind dünn und nicht
aussagekräftig für die heutige Spielstärke. Jedes Spiel erhält zusätzlich einen
**K-Faktor** basierend auf der Turnierwichtigkeit: ein WM-Spiel bewegt die
Elo-Bewertungen 3-mal stärker als ein Freundschaftsspiel. Folgt der
[World Football Elo Ratings](https://www.eloratings.net/about) Konvention.

In [4]:
# Filter to modern era (1990+) — older data is sparse and not very predictive
matches = results[results['date'] >= '1990-01-01'].copy().reset_index(drop=True)
matches = matches.dropna(subset=['home_score', 'away_score'])
print(f"Modern era matches: {len(matches):,}")

# Tournament importance weights (K-factor multipliers)
# Higher weight = match has more impact on Elo
TOURNAMENT_WEIGHT = {
    'FIFA World Cup': 60,
    'FIFA World Cup qualification': 40,
    'UEFA Euro': 50,
    'UEFA Euro qualification': 35,
    'Copa América': 50,
    'African Cup of Nations': 40,
    'AFC Asian Cup': 40,
    'UEFA Nations League': 35,
    'Friendly': 20,
}
DEFAULT_WEIGHT = 30  # for any tournament not listed

def get_weight(tournament):
    return TOURNAMENT_WEIGHT.get(tournament, DEFAULT_WEIGHT)

matches['k_factor'] = matches['tournament'].apply(get_weight)
print("\nSample tournaments and their weights:")
print(matches.groupby('tournament')['k_factor'].first().sort_values(ascending=False).head(10))

Modern era matches: 32,260

Sample tournaments and their weights:
tournament
FIFA World Cup                          60
Copa América                            50
UEFA Euro                               50
AFC Asian Cup                           40
FIFA World Cup qualification            40
African Cup of Nations                  40
UEFA Euro qualification                 35
UEFA Nations League                     35
Morocco, Capital of African Football    30
Mukuru 4 Nations                        30
Name: k_factor, dtype: int64


## 3. Compute Elo ratings / Berechnung der Elo-Bewertungen

Walk through every match chronologically, updating each team's rating after the
result. Key components:

- **Home advantage** (+65 Elo): applied only when the venue is non-neutral
- **Goal difference multiplier**: bigger wins move ratings more (capped to avoid blowout dominance)
- **Pre-match ratings stored**: critical for leakage-free backtesting in Phase 2

The output is a per-team rating reflecting their strength as of June 2026.

---

Durchlaufen aller Spiele chronologisch und Aktualisierung der Bewertung jeder
Mannschaft nach dem Ergebnis. Hauptkomponenten:

- **Heimvorteil** (+65 Elo): angewendet nur bei nicht-neutralem Spielort
- **Tordifferenz-Multiplikator**: deutlichere Siege bewegen die Bewertung stärker (gedeckelt, um Kantersiege zu begrenzen)
- **Vor-Spiel-Bewertungen gespeichert**: entscheidend für leckagefreies Backtesting in Phase 2

Das Ergebnis ist eine Bewertung pro Mannschaft, die deren Spielstärke im Juni 2026 widerspiegelt.

In [6]:
# Elo rating calculation
# Reference: World Football Elo Ratings methodology

INITIAL_ELO = 1500
HOME_ADVANTAGE = 65  # Elo points added to home team (only for non-neutral venues)

def expected_score(rating_a, rating_b):
    """Probability that team A beats team B based on Elo ratings."""
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def goal_difference_multiplier(goal_diff):
    """Bigger wins move Elo more — World Football Elo formula."""
    gd = abs(goal_diff)
    if gd <= 1:
        return 1.0
    elif gd == 2:
        return 1.5
    else:
        return (11 + gd) / 8


def compute_elo(matches_df):
    """Walk through all matches chronologically and update Elo ratings."""
    matches_df = matches_df.sort_values('date').reset_index(drop=True)
    ratings = {}  # team -> current Elo
    
    # Store pre-match ratings for later analysis
    home_elo_before = np.zeros(len(matches_df))
    away_elo_before = np.zeros(len(matches_df))
    
    for i, row in matches_df.iterrows():
        home, away = row['home_team'], row['away_team']
        
        # Initialize new teams at 1500
        if home not in ratings:
            ratings[home] = INITIAL_ELO
        if away not in ratings:
            ratings[away] = INITIAL_ELO
        
        # Home advantage only applies if venue is not neutral
        home_adj = ratings[home] + (HOME_ADVANTAGE if not row['neutral'] else 0)
        away_adj = ratings[away]
        
        # Save pre-match ratings
        home_elo_before[i] = ratings[home]
        away_elo_before[i] = ratings[away]
        
        # Expected result (probability home team wins)
        expected_home = expected_score(home_adj, away_adj)
        
        # Actual result: 1 if home wins, 0.5 if draw, 0 if loss
        if row['home_score'] > row['away_score']:
            actual_home = 1.0
        elif row['home_score'] == row['away_score']:
            actual_home = 0.5
        else:
            actual_home = 0.0
        
        # Goal difference multiplier amplifies decisive wins
        gd_mult = goal_difference_multiplier(row['home_score'] - row['away_score'])
        
        # K-factor (importance weight from previous cell)
        k = row['k_factor'] * gd_mult
        
        # Update both ratings symmetrically
        delta = k * (actual_home - expected_home)
        ratings[home] += delta
        ratings[away] -= delta
    
    matches_df['home_elo_before'] = home_elo_before
    matches_df['away_elo_before'] = away_elo_before
    return matches_df, ratings


# Run it
matches_with_elo, final_ratings = compute_elo(matches)
print(f"Processed {len(matches_with_elo):,} matches")
print(f"Tracking {len(final_ratings)} teams")

Processed 32,260 matches
Tracking 326 teams


## 4. Where the teams stand / Wo die Mannschaften stehen

A sanity check: do the top-rated teams match what we'd expect, and do the usual
favourites land in roughly the right place?

---

Eine Plausibilitätsprüfung: passen die bestbewerteten Mannschaften zu unseren
Erwartungen, und landen die üblichen Favoriten ungefähr an der richtigen Stelle?

In [7]:
# Convert final ratings to a sorted dataframe
elo_df = (
    pd.DataFrame(list(final_ratings.items()), columns=['team', 'elo'])
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
elo_df.index += 1  # rank starts at 1

print("Top 20 teams by Elo (current):")
print(elo_df.head(20).to_string())

print("\nWhere are the favourites?")
favourites = ['Argentina', 'France', 'Spain', 'Brazil', 'Germany', 'England',
              'Portugal', 'Netherlands', 'Italy', 'Belgium', 'Croatia',
              'United States', 'Mexico', 'Canada', 'Japan', 'Morocco']
print(elo_df[elo_df['team'].isin(favourites)].to_string())

Top 20 teams by Elo (current):
           team          elo
1         Spain  2171.784594
2     Argentina  2149.755469
3        France  2081.969871
4       England  2046.479342
5        Brazil  2031.500940
6      Colombia  2026.997937
7      Portugal  1992.888821
8       Ecuador  1990.101707
9         Japan  1985.855691
10  Netherlands  1978.753437
11      Germany  1977.423092
12      Morocco  1974.844215
13      Uruguay  1937.961478
14       Mexico  1933.028861
15      Croatia  1931.421657
16       Norway  1930.817774
17       Turkey  1926.172246
18  Switzerland  1924.600607
19      Belgium  1921.513746
20      Senegal  1894.626362

Where are the favourites?
             team          elo
1           Spain  2171.784594
2       Argentina  2149.755469
3          France  2081.969871
4         England  2046.479342
5          Brazil  2031.500940
7        Portugal  1992.888821
9           Japan  1985.855691
10    Netherlands  1978.753437
11        Germany  1977.423092
12        Morocco  1974

## 5. Turning ratings into match probabilities / Von Bewertungen zu Spielwahrscheinlichkeiten

A simple function that takes two team names and returns the win/draw/loss
probability for each. The draw heuristic (`0.28 × exp(-rating_diff/400)`) is
a documented Elo limitation. Elo is a strength-comparison model and doesn't
naturally produce draw probabilities. We'll come back to this in Phase 2's
calibration analysis.

Example: a sample of interesting opening-round group-stage matches.

---

Eine einfache Funktion, die zwei Mannschaftsnamen entgegennimmt und die
Sieg/Unentschieden/Niederlage-Wahrscheinlichkeit für jede zurückgibt. Die
Unentschieden-Heuristik (`0.28 × exp(-rating_diff/400)`) ist eine dokumentierte
Einschränkung des Elo-Modells. Elo ist ein Stärkevergleichsmodell und liefert
keine natürlichen Unentschieden-Wahrscheinlichkeiten. Darauf kommen wir in der
Kalibrierungsanalyse von Phase 2 zurück.

Beispiel: eine Auswahl interessanter Spiele der ersten Gruppenphase-Runde.

In [8]:
def predict_match(home_team, away_team, ratings, neutral=True):
    """Return (P_home_win, P_draw, P_away_win) based on Elo.
    Note: pure Elo gives win probability — we approximate draw using a simple heuristic.
    """
    r_home = ratings.get(home_team, INITIAL_ELO)
    r_away = ratings.get(away_team, INITIAL_ELO)
    
    home_adj = r_home + (HOME_ADVANTAGE if not neutral else 0)
    
    # Probability home is "stronger" (without draws)
    p_home_strict = expected_score(home_adj, r_away)
    
    # Approximate draw probability — depends on closeness of ratings
    # Heuristic: draws are most likely when teams are evenly matched
    rating_diff = abs(home_adj - r_away)
    p_draw = 0.28 * np.exp(-rating_diff / 400)  # ~28% for equal teams, decays for mismatches
    
    # Distribute the remaining probability
    p_home = p_home_strict * (1 - p_draw)
    p_away = (1 - p_home_strict) * (1 - p_draw)
    
    return p_home, p_draw, p_away


# Some interesting Group Stage matches
matches_to_predict = [
    ('Mexico', 'South Africa', False),    # opener, Mexico hosts
    ('Germany', 'Curacao', True),
    ('Argentina', 'Algeria', True),
    ('Spain', 'Cape Verde', True),
    ('France', 'Senegal', True),
    ('Brazil', 'Morocco', True),
    ('United States', 'Paraguay', False), # USA hosts
    ('Netherlands', 'Japan', True),
]

print(f"{'Match':<40} {'P(Home)':>8} {'P(Draw)':>8} {'P(Away)':>8}")
print('-' * 70)
for home, away, neutral in matches_to_predict:
    p_home, p_draw, p_away = predict_match(home, away, final_ratings, neutral=neutral)
    match_str = f"{home} vs {away}" + ("" if neutral else " (H)")
    print(f"{match_str:<40} {p_home:>7.1%} {p_draw:>7.1%} {p_away:>7.1%}")

Match                                     P(Home)  P(Draw)  P(Away)
----------------------------------------------------------------------
Mexico vs South Africa (H)                 77.4%   11.9%   10.7%
Germany vs Curacao                         86.0%    8.5%    5.5%
Argentina vs Algeria                       72.7%   13.6%   13.7%
Spain vs Cape Verde                        88.2%    7.5%    4.3%
France vs Senegal                          61.5%   17.5%   20.9%
Brazil vs Morocco                          44.0%   24.3%   31.7%
United States vs Paraguay (H)              37.0%   27.5%   35.5%
Netherlands vs Japan                       35.5%   27.5%   37.0%


## 6. Save model artifacts / Modellartefakte speichern

Persist the trained model so Phase 2 (backtest) and Phase 3 (Monte Carlo) can
load it without retraining.

---

Speichert das trainierte Modell, damit Phase 2 (Backtest) und Phase 3 (Monte Carlo)
es ohne erneutes Training laden können.

In [11]:
import json
from pathlib import Path

# Create a processed data folder
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Save match-level data with pre-match Elo ratings (useful for backtesting later)
matches_with_elo.to_csv("../data/processed/matches_with_elo.csv", index=False)

# Save current Elo ratings as JSON (small, human-readable, easy to load)
elo_dict = {team: round(rating, 2) for team, rating in final_ratings.items()}
with open("../data/processed/elo_ratings.json", "w") as f:
    json.dump(elo_dict, f, indent=2, sort_keys=True)

print(f"Saved {len(matches_with_elo):,} matches with pre-match Elo to CSV")
print(f"Saved {len(elo_dict)} team ratings to JSON")
print(f"\nFiles in data/processed:")
for f in Path("../data/processed").iterdir():
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.1f} KB)")

Saved 32,260 matches with pre-match Elo to CSV
Saved 326 team ratings to JSON

Files in data/processed:
  elo_ratings.json (7.7 KB)
  matches_with_elo.csv (3969.1 KB)
